# Results for model: utter-project/eurollm-9b-instruct

In [ ]:
import polars as pl
import pandas as pd
import lightgbm as lgb
from scipy.stats import top_quantile
from dateutil.relativedelta import relativedelta

# Step 1: Define data path and feature list
data_path = './jane_street_data/train.parquet'
feature_list = ['feature_00']
response_column = 'responder_6'
lags = [relativedelta(days=i) for i in range(-30, 0)]

# Step 2: Load data using Polars
data_df = pl.read_parquet(data_path)
data_df['date_id'] = pd.to_datetime(data_df['date_id']).dt.dayofweek

# Step 3: Feature Engineering
data_df['feature_00'] = data_df[feature_list[0]] / data_df[response_column].quantile(top_quantile).to_pandas().rolling(w=len(lags)).apply(lambda x: select(('feature_00', select(('feature_00'))).shift(-top_quantile(x, w=lags))), na_policy='propagate', meta=(<type 'list'>, <type 'list'>))

# Step 4: Train an XGBoost Regressor
xg_reg = lgb.LGBMRegressor(n_estimators=1000)
xg_reg.fit(data_df[feature_list + ['feature_00', 'date_id']], data_df[response_column])

# Save the trained model
lbfgs = MultiLBFGS()
multi_lbfgs = MultiLBFGS(max_nfev=10000, line_search_fn='strong_wolfe', avoid_unnecessary_updates_fn=None, lbfgs=lbfgs, max_linesearch_iter=10, max_evaluations=None, accept_improvement_threshold=-5e-08, min_ancestor_improvement_threshold=1e-08)

final_reg = lgb.LGBMRegressor(boosting_type="gbdt", learning_rate=0.1, n_estimators=10000, num_leaves=31, max_depth=-1, feature_fraction=0.95, bagging_fraction=0.85, bagging_freq=5, min_data_in_leaf=20, min_sum_hessian_in_leaf=5.0, min_split_gain_ratio=0.01, min_gain_ratio=0.01, verbose=-1, num_threads=8, feature_fraction_base_jitter=0.05, bagging_freq_leaf=0)
final_reg.fit(X_train, y_train, callbacks=[multi_lbfgs], evals=[(X_val, y_val)], num_boost_round=105000)

export_dir = './model/lgbm_model_not_yet_exported/'
lgb.plot_importance(final_reg)
lgb.dump_model(final_reg, export_dir + 'lgbm_model.txt')

# Step 5: Save the trained model
final_reg.save_model(export_dir + 'lgbm_model.txt')

# Step 6: Fit and predict
fin_reg = lgb.Booster(model_file=export_dir + 'lgbm_model.txt')
fin_reg.predict(data_df[feature_list])
fin_reg.feature_importances_